In [ ]:

import json
import os
import time
import tracemalloc
from pathlib import Path

import pandas as pd
import psutil

from syngps.utils import (
    build_synthgraph_from_json,
    identify_individual_synthesis_routes,
    find_top_n_routes,
)
from syngps.utils.graph_utils import get_leaf_intermediate_substances

### Config
JSON_FOLDER = "../data/graphs/with_inv_file"
TOP_N = 1


In [2]:

def benchmark_graph_file(json_file: Path, top_n: int = 5) -> dict:
    """
    Process a single synthesis graph JSON file and return a metrics dict covering
    graph topology, route statistics, wall-clock timing, CPU time, and memory usage.

    Note: identify_individual_synthesis_routes is called twice per file —
    once here explicitly (for candidate/viable counts) and once internally inside
    find_top_n_routes. Both calls are included in the reported times to reflect
    realistic end-to-end cost.
    """
    process = psutil.Process(os.getpid())

    tracemalloc.start()
    cpu_before = process.cpu_times()
    rss_before = process.memory_info().rss

    # ── Load & build ──────────────────────────────────────────────────────────
    t0 = time.perf_counter()
    with open(json_file) as f:
        data = json.load(f)
    synth_graph = build_synthgraph_from_json(data)
    t1 = time.perf_counter()

    n_nodes = synth_graph.synthesis_graph.number_of_nodes()
    n_edges = synth_graph.synthesis_graph.number_of_edges()
    target = synth_graph.target_molecule_node_id

    # ── Leaf intermediate availability ────────────────────────────────────────
    avail_map = {a["inchikey"]: a for a in (data.get("availability") or [])}

    def is_available(ik):
        a = avail_map.get(ik, {})
        return (
            a.get("inventory", {}).get("available", False)
            or a.get("commercial_availability", {}).get("available", False)
        )

    leaf_ims = get_leaf_intermediate_substances(synth_graph.synthesis_graph)
    n_leaf_unavail = sum(
        1 for n in leaf_ims
        if not is_available(synth_graph.synthesis_graph.nodes[n].get("inchikey", ""))
    )

    # ── Route identification (for candidate / viable counts) ──────────────────
    t2 = time.perf_counter()
    viable_raw, candidates_raw, combo_graphs = identify_individual_synthesis_routes(
        synth_graph.synthesis_graph,
        synth_graph.target_molecule_node_id,
    )
    t3 = time.perf_counter()

    # ── Top-N route ranking ───────────────────────────────────────────────────
    t4 = time.perf_counter()
    routes = find_top_n_routes(synth_graph, top_n)
    t5 = time.perf_counter()

    # ── Finalise measurements ─────────────────────────────────────────────────
    _, peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    cpu_after = process.cpu_times()
    rss_after = process.memory_info().rss

    return {
        # Identity
        "file": json_file.name,
        "target_molecule": target,
        # Graph topology
        "nodes": n_nodes,
        "edges": n_edges,
        "leaf_ims_total": len(leaf_ims),
        "leaf_ims_available": len(leaf_ims) - n_leaf_unavail,
        "leaf_ims_unavailable": n_leaf_unavail,
        # Route statistics
        "combination_graphs": len(combo_graphs),
        "route_candidates": len(candidates_raw),
        "viable_routes": len(viable_raw),
        "top_n_config": top_n,
        "top_n_returned": len(routes),
        # Timing (seconds)
        "build_time_s": round(t1 - t0, 4),
        "identify_time_s": round(t3 - t2, 4),
        "find_top_n_time_s": round(t5 - t4, 4),
        "total_time_s": round(t5 - t0, 4),
        # CPU time (seconds)
        "cpu_user_s": round(cpu_after.user - cpu_before.user, 4),
        "cpu_system_s": round(cpu_after.system - cpu_before.system, 4),
        # Memory
        "peak_alloc_mb": round(peak_bytes / 1024**2, 3),
        "rss_delta_mb": round((rss_after - rss_before) / 1024**2, 3),
    }


In [3]:

json_path = Path(JSON_FOLDER)
json_files = sorted(json_path.glob("*.json"))
print(f"Found {len(json_files)} JSON files in {json_path.resolve()}\n")

records = []
errors = []

for i, jf in enumerate(json_files, 1):
    try:
        rec = benchmark_graph_file(jf, top_n=TOP_N)
        records.append(rec)
        print(
            f"[{i:>3}/{len(json_files)}] {jf.name:<55} "
            f"{rec['total_time_s']:>6.3f}s  "
            f"nodes={rec['nodes']:>4}  edges={rec['edges']:>4}  "
            f"viable={rec['viable_routes']:>2}  top_n={rec['top_n_returned']:>2}  "
            f"peak={rec['peak_alloc_mb']:>6.2f}MB"
        )
    except Exception as exc:
        errors.append({"file": jf.name, "error": str(exc)})
        print(f"[{i:>3}/{len(json_files)}] ERROR  {jf.name}: {exc}")

print(f"\n✓ Processed {len(records)} files successfully, {len(errors)} errors.")


Found 160 JSON files in /Users/millernar/Development/syngps/revision/data/graphs/with_inv_file

[  1/160] AISQMYCMHKTHHN-SSEXGKCCSA-N_response.json                0.009s  nodes=   4  edges=   3  viable= 0  top_n= 0  peak=  0.10MB
[  2/160] ALQUTEKNDPODSS-UHFFFAOYSA-N_response.json                0.008s  nodes=   7  edges=   6  viable= 0  top_n= 0  peak=  0.09MB
[  3/160] ASSZZQOXRJNXNL-NRFANRHFSA-N_response.json                0.010s  nodes=  21  edges=  22  viable= 0  top_n= 0  peak=  0.24MB
[  4/160] AWDRDIZACXEMRA-UHFFFAOYSA-N_response.json                0.009s  nodes=  12  edges=  11  viable= 0  top_n= 0  peak=  0.12MB
[  5/160] AXZKSUNGRDJIFL-UHFFFAOYSA-N_response.json                0.010s  nodes=  27  edges=  36  viable= 0  top_n= 0  peak=  0.18MB
[  6/160] BFIQWRSPSDOJCV-UHFFFAOYSA-N_response.json                0.008s  nodes=  14  edges=  13  viable= 0  top_n= 0  peak=  0.13MB
[  7/160] BHKKSKOHRFHHIN-MRVPVSSYSA-N_response.json                0.011s  nodes=  23  edges=  26  v

In [ ]:

df = pd.DataFrame(records)

# Ensure consistent column ordering
col_order = [
    "file", "target_molecule",
    "nodes", "edges",
    "leaf_ims_total", "leaf_ims_available", "leaf_ims_unavailable",
    "combination_graphs", "route_candidates", "viable_routes",
    "top_n_config", "top_n_returned",
    "build_time_s", "identify_time_s", "find_top_n_time_s", "total_time_s",
    "cpu_user_s", "cpu_system_s",
    "peak_alloc_mb", "rss_delta_mb",
]
df = df[[c for c in col_order if c in df.columns]]

print(f"DataFrame shape: {df.shape}")
df.head(10)


In [ ]:

# Descriptive statistics for all numeric columns
numeric_cols = [
    "nodes", "edges",
    "leaf_ims_total", "leaf_ims_available", "leaf_ims_unavailable",
    "combination_graphs", "route_candidates", "viable_routes", "top_n_returned",
    "build_time_s", "identify_time_s", "find_top_n_time_s", "total_time_s",
    "cpu_user_s", "cpu_system_s",
    "peak_alloc_mb", "rss_delta_mb",
]
df[numeric_cols].describe().round(4)


In [ ]:

# ── Timing breakdown: which phase dominates? ──────────────────────────────────
timing_summary = df[["build_time_s", "identify_time_s", "find_top_n_time_s", "total_time_s"]].agg(
    ["mean", "median", "max", "sum"]
).round(4)
print("=== Timing breakdown (seconds) ===")
print(timing_summary.to_string())

# ── Graphs with viable routes vs fully disqualified ───────────────────────────
print("\n=== Route viability summary ===")
viability = df["viable_routes"].apply(lambda v: "has_viable" if v > 0 else "all_disqualified").value_counts()
print(viability.to_string())

# ── Correlation: does graph size predict processing time? ────────────────────
print("\n=== Correlation with total_time_s ===")
corr_cols = ["nodes", "edges", "combination_graphs", "route_candidates", "viable_routes", "total_time_s"]
print(df[corr_cols].corr()[["total_time_s"]].round(4).to_string())

# ── Top 10 slowest graphs ─────────────────────────────────────────────────────
print("\n=== Top 10 slowest graphs ===")
slowest = df.nlargest(10, "total_time_s")[["file", "nodes", "edges", "viable_routes", "total_time_s", "peak_alloc_mb"]]
print(slowest.to_string(index=False))
